# 03 — A publication-ready figure

This notebook builds a polished **multi-model comparison figure** from REF metric values
and saves it in formats suitable for a paper or report.

**Prerequisites:** [01 — REF concepts](01-ref-concepts.ipynb), [02 — Querying the REF API](02-querying-the-api.ipynb).

**What you need:** an internet connection.

## Goal

We want a single figure that compares many CMIP models on one scalar diagnostic — the
kind of figure you would put in a paper. We will:

1. fetch scalar metric values for a diagnostic;
2. tidy them into a `DataFrame`;
3. apply a consistent publication plotting style;
4. build and save the figure.

In [ ]:
from ref_tutorials import (
    get_client,
    metric_values_to_dataframe,
    model_comparison_figure,
    save_figure,
    set_publication_style,
)

client = get_client()
set_publication_style()

## 1–2. Fetch and tidy the metric values

We pick a diagnostic that exposes scalar values and load them into a tidy frame.

In [ ]:
from climate_ref_client.api.diagnostics import (
    diagnostics_list,
    diagnostics_list_metric_values,
)
from climate_ref_client.models.metric_value_type import MetricValueType

diagnostics = diagnostics_list.sync(client=client).data
diagnostic = next(d for d in diagnostics if d.execution_groups)

values = diagnostics_list_metric_values.sync(
    diagnostic.provider.slug,
    diagnostic.slug,
    value_type=MetricValueType.SCALAR,
    client=client,
).data

df = metric_values_to_dataframe(values)
df.head()

A diagnostic often reports several *statistics*. For a clean comparison figure we focus
on one statistic, and on the metric values that carry a `source_id` (the model name).

In [ ]:
subset = df.dropna(subset=["source_id", "value"]).copy()

if "statistic" in subset.columns:
    statistic = subset["statistic"].dropna().iloc[0]
    subset = subset[subset["statistic"] == statistic]
else:
    statistic = diagnostic.name

print(f"Statistic: {statistic}")
print(f"Models:    {subset['source_id'].nunique()}")
subset.head()

## 3–4. Build the figure

`model_comparison_figure` draws one bar per model, sorted by value. Where a model has
several ensemble members it is aggregated to its mean and the spread is shown as an
error bar. The figure already uses the publication style we applied above.

In [ ]:
fig, ax = model_comparison_figure(
    subset,
    value_col="value",
    model_col="source_id",
    title=f"{diagnostic.name}",
    ylabel=str(statistic),
)
fig

The returned `fig` and `ax` are ordinary matplotlib objects, so you can keep tweaking —
add a reference line, annotate, change colours — before saving.

## Save the figure

`save_figure` writes both a `.png` (quick viewing) and a `.pdf` (vector, for papers)
at 300 dpi.

In [ ]:
written = save_figure(fig, "output/model-comparison")
written

## Adapting this recipe

To make the same figure for a different diagnostic, change the `diagnostic` selection in
step 1. To compare a different quantity, change `statistic`. The helper functions
(`metric_values_to_dataframe`, `model_comparison_figure`, `save_figure`) stay the same —
see `src/ref_tutorials/` if you want to see how they work or extend them.

**Next:** [04 — Running a diagnostic locally](04-local-diagnostic-run.ipynb).